# Executable LSTM topic-classification pipeline

This notebook is an execution interface. All reusable logic lives in `src/topic_classifier/`.

## Goal

Train and evaluate an end-to-end LSTM classifier for four product-review topics.

## Setup

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Run the notebook from inside the repository.')

sys.path.insert(0, str(ROOT))

from src.topic_classifier import PipelineConfig, run_pipeline

print('Environment configured.')

Environment configured.


## Steps

### 1. Configure and run

The number of epochs can be controlled with the `PIPELINE_EPOCHS` environment variable.

In [2]:
epochs = int(os.getenv('PIPELINE_EPOCHS', '20'))
config = PipelineConfig(
    dataset_path=ROOT / 'data' / 'topic_samples.csv',
    epochs=epochs,
    seed=42,
)

result = run_pipeline(config)
result

PipelineResult(dataset_size=80, train_size=64, test_size=16, labels=['refrigerator', 'smartphone', 'television', 'washing_machine'], metrics={'loss': 0.9746034145355225, 'accuracy': 0.6875}, history={'accuracy': [0.203125, 0.40625, 0.515625, 0.515625, 0.515625, 0.5625, 0.578125, 0.59375, 0.609375, 0.625, 0.65625, 0.71875, 0.8125, 0.890625, 0.890625, 0.9375, 0.96875, 0.984375, 1.0, 1.0], 'loss': [1.3867290019989014, 1.3821492195129395, 1.3775653839111328, 1.3719476461410522, 1.3644983768463135, 1.3546133041381836, 1.341589331626892, 1.3241366147994995, 1.3008067607879639, 1.269571304321289, 1.22808837890625, 1.1728817224502563, 1.101847529411316, 1.0170207023620605, 0.9265625476837158, 0.838168740272522, 0.7558212876319885, 0.679690957069397, 0.6070840954780579, 0.5338726043701172]}, confusion_matrix=[[2, 1, 0, 1], [0, 2, 0, 2], [0, 0, 3, 1], [0, 0, 0, 4]], predictions=[{'text': 'the fingerprint reader stopped recognizing me', 'expected': 'smartphone', 'predicted': 'washing_machine'}, {

### 2. Resultados

In [3]:
summary = {
    'dataset_size': result.dataset_size,
    'train_size': result.train_size,
    'test_size': result.test_size,
    'labels': result.labels,
    'epochs': epochs,
    'test_accuracy': round(result.metrics['accuracy'], 4),
    'test_loss': round(result.metrics['loss'], 4),
    'confusion_matrix': result.confusion_matrix,
}

summary

{'dataset_size': 80,
 'train_size': 64,
 'test_size': 16,
 'labels': ['refrigerator', 'smartphone', 'television', 'washing_machine'],
 'epochs': 20,
 'test_accuracy': 0.6875,
 'test_loss': 0.9746,
 'confusion_matrix': [[2, 1, 0, 1], [0, 2, 0, 2], [0, 0, 3, 1], [0, 0, 0, 4]]}

In [4]:
result.predictions[:8]

[{'text': 'the fingerprint reader stopped recognizing me',
  'expected': 'smartphone',
  'predicted': 'washing_machine'},
 {'text': 'the television has sound but no picture',
  'expected': 'television',
  'predicted': 'television'},
 {'text': 'the refrigerator water dispenser stopped working',
  'expected': 'refrigerator',
  'predicted': 'refrigerator'},
 {'text': 'the refrigerator takes too long to reach temperature',
  'expected': 'refrigerator',
  'predicted': 'smartphone'},
 {'text': 'the touchscreen does not respond to commands',
  'expected': 'smartphone',
  'predicted': 'washing_machine'},
 {'text': 'the washing machine does not complete the spin cycle',
  'expected': 'washing_machine',
  'predicted': 'washing_machine'},
 {'text': 'the wash cycle takes too long',
  'expected': 'washing_machine',
  'predicted': 'washing_machine'},
 {'text': 'the television remote control stopped working',
  'expected': 'television',
  'predicted': 'television'}]

## Checks

In [5]:
expected_labels = {'smartphone', 'television', 'refrigerator', 'washing_machine'}

assert result.dataset_size == 80
assert result.train_size == 64
assert result.test_size == 16
assert set(result.labels) == expected_labels
assert len(result.predictions) == result.test_size
assert result.metrics['accuracy'] >= 0.50, result.metrics

{'status': 'ok', 'accuracy': round(result.metrics['accuracy'], 4)}

{'status': 'ok', 'accuracy': 0.6875}

## Next Steps

Replace the synthetic dataset with real, labeled, PII-free reviews. Preserve the `text,topic` schema to reuse the entire pipeline.